In [1]:
from utils import *
import json
ROOT.resolve()

PosixPath('/Users/lukestrange/Code/bus-tracking')

In [2]:
agencies, routes, trips, stops, stop_times, calendar, calendar_dates, shapes, feed_info = load_full_gtfs(ROOT / "war/yorkshire/190924", include=['shapes.txt', 'feed_info.txt'])

In [3]:
tt_agencies, tt_routes, tt_trips, tt_stops, tt_stop_times, tt_calendar, tt_calendar_dates = load_full_gtfs(ROOT / "18SepGB_GTFS_Timetables_Downloaded/yorkshire")

Get an example data file for the website. Use no.33 bus.

In [4]:
agency_id = 'OP931'
bus_num = '12'
date_str = "2024-09-19"

In [5]:
route_id = routes[(routes.agency_id == agency_id) & (routes.route_short_name == bus_num)].route_id.values[0]

In [6]:
trips_this_route = trips[trips.route_id == route_id][['trip_id', 'trip_headsign', 'shape_id']]
unique_trips = trips_this_route.trip_id.unique()
unique_shapes = trips_this_route.shape_id.unique()
trips_this_route.count()

trip_id          128
trip_headsign    128
shape_id         128
dtype: int64

Get the timetabled trips that match the real trips

In [7]:
matched_tt_stop_times = tt_stop_times[tt_stop_times.trip_id.isin(unique_trips)]
matched_tt_stop_times_simple = matched_tt_stop_times.loc[:, ['trip_id', 'arrival_time', 'stop_id', 'stop_sequence']]
matched_tt_stop_times_simple.rename(columns={'arrival_time': 'timetable'}, inplace=True)
matched_tt_stop_times_simple

,trip_id,timetable,stop_id,stop_sequence
1813045,VJ27aef674e6edaf72468f06947708be0b2fa16bfd,04:44:00,450010945,0
1813046,VJ27aef674e6edaf72468f06947708be0b2fa16bfd,04:45:19,450013674,1
1813047,VJ27aef674e6edaf72468f06947708be0b2fa16bfd,04:46:32,450010311,2
1813048,VJ27aef674e6edaf72468f06947708be0b2fa16bfd,04:47:16,450010312,3
1813049,VJ27aef674e6edaf72468f06947708be0b2fa16bfd,04:47:46,450010313,4
...,...,...,...,...
1825906,VJb15d3193c24466b17486e14ecc0b3a3c8fda2c64,16:23:42,450011480,28
1825907,VJb15d3193c24466b17486e14ecc0b3a3c8fda2c64,16:24:40,450010070,29
1825908,VJb15d3193c24466b17486e14ecc0b3a3c8fda2c64,16:25:32,450012124,30
1825909,VJb15d3193c24466b17486e14ecc0b3a3c8fda2c64,16:26:44,450012125,31


In [8]:
shapes_this_route = shapes[shapes.shape_id.isin(unique_shapes)][['shape_id', 'shape_pt_lon', 'shape_pt_lat']]
shapes_this_route = shapes_this_route.groupby('shape_id').apply(lambda x: x[['shape_pt_lon', 'shape_pt_lat']].values.round(5).tolist(), include_groups=False).reset_index(name='geometry')
line = shapes_this_route.set_index('shape_id')['geometry'].to_dict()

In [9]:
stop_times_this_route = stop_times[stop_times.trip_id.isin(unique_trips)][['trip_id', 'arrival_time', 'stop_id', 'stop_sequence']]
unique_stops = stop_times_this_route.stop_id.unique()
stop_times_this_route.rename(columns={'arrival_time': 'real'}, inplace=True)

# Add the timetabled times
stop_times_this_route = stop_times_this_route.merge(matched_tt_stop_times_simple, on=['trip_id', 'stop_id', 'stop_sequence'], how='inner')

# Convert times to UTC.
stop_times_this_route['real'] = convert_to_unix_timestamp(stop_times_this_route, 'real', date_str)
stop_times_this_route['timetable'] = convert_to_unix_timestamp(stop_times_this_route, 'timetable', date_str)


stop_times_this_route
# Create an empty list to store results
trip_list = []

# Iterate over each unique trip_id
for trip_id in stop_times_this_route['trip_id'].unique():
    
    # Filter rows for the current trip_id
    trip_df = stop_times_this_route[stop_times_this_route['trip_id'] == trip_id]
    # Sort by 'real' time
    trip_df = trip_df.sort_values(by='real')
    # print(trip_df.to_csv())
    # Create a list of dicts for this trip
    current_trip_data = []
    for i, row in trip_df.iterrows():
        trip_data = {
                "stop_id": row['stop_id'],
                "real": int(row['real']),
                "timetable": int(row['timetable'])
            }
        current_trip_data.append(trip_data)
    # Append this trip's list to the main list
    trip_list.append(current_trip_data)

# Output the final list of lists
print(trip_list)

[[{'stop_id': '450010945', 'real': 1726718522, 'timetable': 1726718640}, {'stop_id': '450010313', 'real': 1726718829, 'timetable': 1726718866}, {'stop_id': '450010314', 'real': 1726718866, 'timetable': 1726718906}, {'stop_id': '450023384', 'real': 1726718946, 'timetable': 1726718940}, {'stop_id': '450010315', 'real': 1726719002, 'timetable': 1726719020}, {'stop_id': '450010316', 'real': 1726719029, 'timetable': 1726719075}, {'stop_id': '450023390', 'real': 1726719110, 'timetable': 1726719196}, {'stop_id': '450010304', 'real': 1726719188, 'timetable': 1726719313}, {'stop_id': '450010303', 'real': 1726719254, 'timetable': 1726719353}, {'stop_id': '450023950', 'real': 1726719303, 'timetable': 1726719487}, {'stop_id': '450012974', 'real': 1726719358, 'timetable': 1726719574}, {'stop_id': '450012984', 'real': 1726719485, 'timetable': 1726719660}, {'stop_id': '450011507', 'real': 1726719724, 'timetable': 1726719766}, {'stop_id': '450010621', 'real': 1726719835, 'timetable': 1726719847}, {'st

In [10]:
stops_this_route = stops[stops.stop_id.isin(unique_stops)][['stop_id', 'stop_name', 'stop_lon', 'stop_lat']]
stop_bearings = get_stop_names_and_bearings()[['stop_id', 'Bearing']]

stops_this_route = stops_this_route.merge(stop_bearings, on='stop_id', how='inner')
stops_this_route.rename(columns={'stop_name': 'name', 'stop_lat': 'lat', 'stop_lon': 'lon', 'Bearing': 'bearing'}, inplace=True)
stops_this_route['bearing'] = stops_this_route['bearing'].astype(int)
stops_this_route.set_index('stop_id', inplace=True)
stops = stops_this_route.to_dict(orient='index')
stops

{'450010005': {'name': 'Bodmin Road',
  'lon': -1.559264,
  'lat': 53.752098,
  'bearing': 180},
 '450010010': {'name': 'Middleton Centre A',
  'lon': -1.533956,
  'lat': 53.749111,
  'bearing': 270},
 '450010011': {'name': 'Lingwell Road',
  'lon': -1.542365,
  'lat': 53.748314,
  'bearing': 270},
 '450010017': {'name': 'Middleton Pk Circus',
  'lon': -1.549205,
  'lat': 53.748207,
  'bearing': 270},
 '450010028': {'name': 'Playfair Road',
  'lon': -1.5305,
  'lat': 53.773464,
  'bearing': 0},
 '450010029': {'name': 'Moor Road',
  'lon': -1.530869,
  'lat': 53.77166,
  'bearing': 180},
 '450010030': {'name': 'Moor Road',
  'lon': -1.531046,
  'lat': 53.772099,
  'bearing': 0},
 '450010031': {'name': 'Belle Isle Circus',
  'lon': -1.528648,
  'lat': 53.765446,
  'bearing': 180},
 '450010032': {'name': 'Middleton Road',
  'lon': -1.526578,
  'lat': 53.75787,
  'bearing': 180},
 '450010033': {'name': 'Lanshaw View',
  'lon': -1.530124,
  'lat': 53.752769,
  'bearing': 225},
 '450010034':

In [11]:
content = dict({'line': line, 'stops': stops, 'trips': trip_list})

In [12]:
# content = f'line:{line.strip()},stops:{stops.strip()},trips:{trip_list}'
with open(ROOT / f"web/{bus_num}.json", "w") as f:
    json.dump(content, f, separators=(',',':'))
    # json.dump(line, f, se)